In [2]:
# =========================================================================
# PURPOSE
#   Produce the master-thesis figure that contrasts price levels with
#   their log-returns and overlays a 3-month (63 trading days) rolling
#   standard deviation on each panel. Two assets are plotted: the EU
#   Allowance (EUA) carbon-futures price in USD and the S&P Global
#   LargeMidCap PAB ESG index in USD. The result is a 2x2 grid:
#
#       row 1 : EUA   prices |   EUA   log-returns
#       row 2 : ESG   prices |   ESG   log-returns
#
#   Each panel uses two y-axes: the level/return on the left (blue) and
#   the rolling standard deviation on the right (red).
#
#   External dependencies:
#       - Data.xlsx (input data; sheet 'Tabelle1' with column 'Date',
#         'EU Allowance EUA Futures (USD)' and
#         'S&P Global LargeMidCap PAB ESG Index (USD)')
# =========================================================================

import numpy as np
import pandas as pd
import warnings
import matplotlib
matplotlib.use('Agg')                           # non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
from Functions import rolling_std, format_panel
warnings.filterwarnings("ignore")

# -------------------------------------------------------------------------
# Load price data
# -------------------------------------------------------------------------
T = pd.read_excel(
    '/Users/johannesfelchner/Desktop/Code/Section 4/Section 4.1/Data.xlsx',
    sheet_name='Tabelle1',
    parse_dates=['Date'],
)
T = T.sort_values('Date').reset_index(drop=True)

dates = T['Date']
eua   = T['EU Allowance EUA Futures (USD)'].values.astype(float)
spg   = T['S&P Global LargeMidCap PAB ESG Index (USD)'].values.astype(float)

# -------------------------------------------------------------------------
# Daily log-returns
# r_t = log(P_t) - log(P_{t-1}), with NaN in the first slot to keep
# the array length equal to that of the price series.
# -------------------------------------------------------------------------
ret_eua = np.concatenate([[np.nan], np.diff(np.log(eua))])
ret_spg = np.concatenate([[np.nan], np.diff(np.log(spg))])

# -------------------------------------------------------------------------
# Rolling standard deviation over 63 trading days (~ 3 months)
# -------------------------------------------------------------------------
win = 63

roll_eua_price = rolling_std(eua,    win)
roll_eua_ret   = rolling_std(ret_eua, win)
roll_spg_price = rolling_std(spg,    win)
roll_spg_ret   = rolling_std(ret_spg, win)

# -------------------------------------------------------------------------
# Colour palette
# -------------------------------------------------------------------------
cBlue = '#1B5E8A'   # main series
cRed  = '#C0392B'   # rolling std
cGrey = '#999999'   # zero line for returns

# -------------------------------------------------------------------------
# 5. Global matplotlib style (serif, thesis-friendly)
# -------------------------------------------------------------------------
plt.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset':  'dejavuserif',
    'font.size':         7,
    'axes.linewidth':    0.4,
    'axes.edgecolor':    '#BBBBBB',
    'xtick.major.width': 0.35, 'ytick.major.width': 0.35,
    'xtick.major.size':  2.2,  'ytick.major.size':  2.2,
    'xtick.direction':   'out', 'ytick.direction':  'out',
    'xtick.color':       '#555555', 'ytick.color':  '#555555',
    'axes.labelcolor':   '#222222', 'text.color':   '#222222',
})

fig, axes = plt.subplots(2, 2, figsize=(7.48, 5.0), facecolor='white')
plt.subplots_adjust(left=0.08, right=0.92, top=0.93, bottom=0.08,
                    hspace=0.35, wspace=0.45)
# -------------------------------------------------------------------------
# Build the four panels
# -------------------------------------------------------------------------
# EUA prices (top-left).
ax1 = axes[0, 0]; ax1r = ax1.twinx()
format_panel(ax1, ax1r, dates, eua, roll_eua_price, 
             'Price', '3-Month Rolling Std. Dev.', 'EUA Futures – Prices', 
             cBlue, cRed, cGrey)

# EUA log-returns (top-right).
ax2 = axes[0, 1]; ax2r = ax2.twinx()
format_panel(ax2, ax2r, dates, ret_eua, roll_eua_ret,
             'Return', '3-Month Rolling Std. Dev.',
             'EUA Futures – Returns', cBlue, cRed, cGrey, is_return=True)

# S&P ESG prices (bottom-left).
ax3 = axes[1, 0]; ax3r = ax3.twinx()
format_panel(ax3, ax3r, dates, spg, roll_spg_price,
             'Price', '3-Month Rolling Std. Dev.',
             'ESG Index – Prices', cBlue, cRed, cGrey)

# S&P ESG log-returns (bottom-right).
ax4 = axes[1, 1]; ax4r = ax4.twinx()
format_panel(ax4, ax4r, dates, ret_spg, roll_spg_ret,
             'Return', '3-Month Rolling Std. Dev.',
             'ESG Index – Returns', cBlue, cRed, cGrey, is_return=True)

# -------------------------------------------------------------------------
# Save to disk (PDF for LaTeX inclusion)
# -------------------------------------------------------------------------

fig.savefig(
    '/Users/johannesfelchner/Desktop/Code/Section 4/Section 4.1/price_return_rolling_std.pdf',
    bbox_inches='tight', facecolor='white', edgecolor='none',
)
print("Price/return figure done.")

Price/return figure done.
